## Task 2: Data Quality Engineering 

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, to_date, to_timestamp

spark = SparkSession.builder \
    .appName("RRSIS") \
    .master("local[*]") \
    .getOrCreate()

# Load Time as STRING this time — do not let Spark infer it as timestamp
df = spark.read.csv(
    "hdfs://localhost:9000/rrsis/data/Road Accident Data.csv",
    header=True,
    inferSchema=True
)

# Fix the mis-inferred Time column: cast back to string, we'll parse manually
df = df.withColumn("Time", col("Time").cast("string"))
df.select("Accident Date", "Time").show(5, truncate=False)

C:\Users\Christian\anaconda3\envs\ml_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


+-------------+-------------------+
|Accident Date|Time               |
+-------------+-------------------+
|1/1/2021     |2026-09-06 15:11:00|
|1/5/2021     |2026-09-06 10:59:00|
|1/4/2021     |2026-09-06 14:19:00|
|1/5/2021     |2026-09-06 08:10:00|
|1/6/2021     |2026-09-06 17:25:00|
+-------------+-------------------+
only showing top 5 rows


### calculating nulls

In [3]:
null_counts = df.select([
    count(
        when(
            col(c).isNull() |
            (col(c).cast("string") == "None") |
            (col(c).cast("string") == ""),
            c
        )
    ).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True, truncate=False)

-RECORD 0----------------------------
 Accident_Index             | 0      
 Accident Date              | 0      
 Month                      | 0      
 Day_of_Week                | 0      
 Year                       | 0      
 Junction_Control           | 0      
 Junction_Detail            | 0      
 Accident_Severity          | 0      
 Latitude                   | 0      
 Light_Conditions           | 0      
 Local_Authority_(District) | 0      
 Carriageway_Hazards        | 302549 
 Longitude                  | 0      
 Number_of_Casualties       | 0      
 Number_of_Vehicles         | 0      
 Police_Force               | 0      
 Road_Surface_Conditions    | 317    
 Road_Type                  | 1534   
 Speed_limit                | 0      
 Time                       | 17     
 Urban_or_Rural_Area        | 0      
 Weather_Conditions         | 6057   
 Vehicle_Type               | 0      



## Dataset description

In [3]:
total = df.count()
distinct_total = df.dropDuplicates().count()
distinct_by_index = df.dropDuplicates(["Accident_Index"]).count()

print("Total rows:", total)
print("Distinct rows (all columns):", distinct_total)
print("Distinct Accident_Index values:", distinct_by_index)

Total rows: 307973
Distinct rows (all columns): 307972
Distinct Accident_Index values: 197644


### Incident count by hour, severity, casualities and year

In [4]:
# Speed limit sanity check
df.groupBy("Speed_limit").count().orderBy("Speed_limit").show(30)

# Severity category check
df.groupBy("Accident_Severity").count().show()

# Casualties/vehicles sanity range
df.select("Number_of_Casualties", "Number_of_Vehicles").describe().show()

# Year range check
df.groupBy("Year").count().orderBy("Year").show()

+-----------+------+
|Speed_limit| count|
+-----------+------+
|         10|     3|
|         15|     2|
|         20|  2899|
|         30|200040|
|         40| 25650|
|         50| 10191|
|         60| 46826|
|         70| 22362|
+-----------+------+

+-----------------+------+
|Accident_Severity| count|
+-----------------+------+
|           Slight|263280|
|            Fatal|  3953|
|          Serious| 40740|
+-----------------+------+

+-------+--------------------+------------------+
|summary|Number_of_Casualties|Number_of_Vehicles|
+-------+--------------------+------------------+
|  count|              307973|            307973|
|   mean|  1.3568819344552931|1.8290629373354157|
| stddev|  0.8158569393614277|0.7104766166622113|
|    min|                   1|                 1|
|    max|                  48|                32|
+-------+--------------------+------------------+

+----+------+
|Year| count|
+----+------+
|2021|163554|
|2022|144419|
+----+------+



## data cleaning

In [4]:
from pyspark.sql.functions import to_date, date_format, trim

# --- Issue 3: parse Accident Date into a real date type ---
df_clean = df.withColumn("Accident_Date_Parsed", to_date(col("Accident Date"), "M/d/yyyy"))

# --- Issue 2: extract only the time-of-day from the corrupted Time column ---
# The date portion is garbage (today's system date), so we only trust HH:mm:ss
df_clean = df_clean.withColumn("Time_of_Day", date_format(col("Time"), "HH:mm:ss"))

# --- Issue 6: drop the 1 exact duplicate row ---
df_clean = df_clean.dropDuplicates()

# --- Issue 5: drop rows with missing Time (only 17 of 307,973 — negligible loss) ---
df_clean = df_clean.filter(col("Time").isNotNull())

# --- Issue 5: impute small true-missing categoricals with "Unknown" instead of dropping ---
# Justification: dropping these rows would discard real casualty/severity data
# just because one contextual field wasn't recorded — imputing preserves the record
# for severity/temporal analysis while flagging the field itself as unreliable.
for c in ["Road_Surface_Conditions", "Road_Type", "Weather_Conditions"]:
    df_clean = df_clean.withColumn(
        c,
        when((col(c).isNull()) | (col(c) == ""), "Unknown").otherwise(col(c))
    )

# --- Issue 4: Carriageway_Hazards "None" is a valid category, not missing — leave as is ---
# (No action needed — this documents the

In [5]:
# Accident-level view: one row per unique accident, for temporal/severity counts
# We take the first row per Accident_Index since core accident attributes
# (date, time, severity, location, casualties) are identical across its vehicle rows
df_accidents = df_clean.dropDuplicates(["Accident_Index"])

print("Vehicle-level rows (df_clean):", df_clean.count())
print("Accident-level rows (df_accidents):", df_accidents.count())

Vehicle-level rows (df_clean): 307955
Accident-level rows (df_accidents): 197644


In [6]:
df_clean.show()

+--------------+-------------+-----+-----------+----+--------------------+--------------------+-----------------+---------+--------------------+--------------------------+-------------------+---------+--------------------+------------------+-------------------+-----------------------+------------------+-----------+-------------------+-------------------+--------------------+--------------------+--------------------+-----------+
|Accident_Index|Accident Date|Month|Day_of_Week|Year|    Junction_Control|     Junction_Detail|Accident_Severity| Latitude|    Light_Conditions|Local_Authority_(District)|Carriageway_Hazards|Longitude|Number_of_Casualties|Number_of_Vehicles|       Police_Force|Road_Surface_Conditions|         Road_Type|Speed_limit|               Time|Urban_or_Rural_Area|  Weather_Conditions|        Vehicle_Type|Accident_Date_Parsed|Time_of_Day|
+--------------+-------------+-----+-----------+----+--------------------+--------------------+-----------------+---------+-------------